In [1]:
# --- [CELL 0]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 1}
import numpy 
import matplotlib.pyplot as plt 
import os 

from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Dense, Flatten, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [2]:
# --- [CELL 1]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 2}
src_path_train = 'data_small/training_set'
src_path_test = 'data_small/test_set'

train_datagen = ImageDataGenerator(
        rescale = 1 / 255.0,

        validation_split = 0.20)

test_datagen = ImageDataGenerator(rescale = 1 / 255.0)

In [3]:
# --- [CELL 2]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 3}
batch_size = 30
train_generator = train_datagen.flow_from_directory(
    directory= src_path_train,
    target_size= (223, 223),
    color_mode= "rgb",
    batch_size= batch_size,
    class_mode= "categorical",
    subset= 'training',
    shuffle= True,
    seed= 40
)
valid_generator = train_datagen.flow_from_directory(
    directory= src_path_train,
    target_size= (223, 223),
    color_mode= "rgb",
    batch_size= batch_size,
    class_mode= "categorical",
    subset= 'validation',
    shuffle= True,
    seed= 40
)

Found 8 images belonging to 2 classes.
Found 2 images belonging to 2 classes.


In [4]:
# --- [CELL 3]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 4}
test_generator = test_datagen.flow_from_directory(
    directory=src_path_test,
    target_size=(223, 223),
    color_mode="rgb",
    batch_size=1,
    class_mode="categorical",
    shuffle=False,
    seed=40
)

Found 8 images belonging to 2 classes.


In [5]:
# --- [CELL 4]: ---
# cell_state: edited
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 5}
# === BEFORE (original) ===
# def prepare_model():
#     model = Sequential()
#     model.add(Conv2D(32,kernel_size=(3,3),activation='relu',input_shape=(100, 100, 3)))
#     model.add(MaxPooling2D(pool_size=(2, 2)))
#     model.add(Flatten())
#     model.add(Dense(16, activation='relu'))
#     model.add(Dense(2, activation='softmax'))
#     model.compile(loss="categorical_crossentropy",optimizer="adam",metrics=['accuracy'])
#     return model
# model = prepare_model()
# model.fit(train_generator,
#                     validation_data = valid_generator,
#                     epochs=5)
# model.evaluate(test_generator)

# === AFTER (edited) ===
def prepare_model():
    model = Sequential()
    model.add(Conv2D(32,kernel_size=(3,3),activation='relu',input_shape=(223, 223, 3)))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Flatten())
    model.add(Dense(16, activation='relu'))
    model.add(Dense(2, activation='softmax'))
    model.compile(loss="categorical_crossentropy",optimizer="adam",metrics=['accuracy'])
    return model
model = prepare_model()
model.fit(train_generator,
                    validation_data = valid_generator,
                    epochs=5)
model.evaluate(test_generator)

Epoch 1/5


/usr/local/lib/python3.10/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 856ms/step - accuracy: 0.7500 - loss: 0.6808

/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.7500 - loss: 0.6808 - val_accuracy: 0.5000 - val_loss: 2.1144
Epoch 2/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 139ms/step - accuracy: 0.8750 - loss: 0.2815 - val_accuracy: 0.5000 - val_loss: 12.4302
Epoch 3/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 127ms/step - accuracy: 0.5000 - loss: 4.1416 - val_accuracy: 0.5000 - val_loss: 4.7990
Epoch 4/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 136ms/step - accuracy: 1.0000 - loss: 0.0070 - val_accuracy: 0.5000 - val_loss: 6.4714
Epoch 5/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 138ms/step - accuracy: 1.0000 - loss: 0.0640 - val_accuracy: 0.5000 - val_loss: 10.1780
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.2180 - loss: 13.6580


[8.418424606323242, 0.5]

In [6]:
# Fails for the buggy model because generator images are 223x223x3 while model expects 100x100x3.
assert model.input_shape[1:] == train_generator.image_shape, (
    f"Model input shape {model.input_shape[1:]} does not match generator shape {train_generator.image_shape}"
)
xb, yb = next(train_generator)
assert xb.shape[1:] == model.input_shape[1:], (
    f"Batch feature shape {xb.shape[1:]} does not match model input shape {model.input_shape[1:]}"
)
assert yb.shape[-1] == model.output_shape[-1], (
    f"Label classes {yb.shape[-1]} do not match model output units {model.output_shape[-1]}"
)